In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [5]:
df = pd.read_csv("../../../data/processed/ml_dataset.csv")

df.head()

,stock_code,trade_date,return_1d,return_5d,return_10d,return_20d,intraday_return,high_low_range,gap,sma_5,...,macd,macd_signal,macd_hist,volatility_5,volatility_20,atr_14,volume_change_1d,volume_sma_20,volume_ratio_20,target_return_5d
0,660,2024-06-11,0.021635,0.094233,0.054591,0.181212,0.011905,0.043689,0.009615,203000.0,...,6187.902397,5195.016563,992.885834,0.031048,0.023996,6757.468051,0.253659,3435519.80,0.893653,0.103529
1,660,2024-06-12,0.011765,0.112261,0.061728,0.169750,0.014151,0.023697,-0.002353,207340.0,...,6911.664995,5538.346249,1373.318746,0.028770,0.023814,6631.934619,-0.299278,3381585.20,0.636190,0.086047
2,660,2024-06-13,0.032558,0.146102,0.096296,0.198057,-0.017699,0.034247,0.051163,213000.0,...,7958.354609,6022.347921,1936.006687,0.026692,0.024432,6979.653575,1.685444,3532295.70,1.635559,0.069820
3,660,2024-06-14,-0.004505,0.065060,0.129280,0.145078,-0.017778,0.041667,0.013514,215700.0,...,8607.944994,6539.467336,2068.477659,0.014806,0.023386,7123.964034,-0.426854,3443697.95,0.961531,0.058824
4,660,2024-06-17,0.009050,0.072115,0.178647,0.174302,0.018265,0.047945,-0.009050,218700.0,...,9178.331251,7067.240119,2111.091132,0.013915,0.022745,7365.109460,-0.336094,3411545.25,0.644382,0.000000


In [6]:
print(df.shape)
print(df.columns.tolist())

(2685, 28)
['stock_code', 'trade_date', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20', 'target_return_5d']


In [8]:
feature_cols = [
    col for col in df.columns
    if col not in ["stock_code", "trade_date", "target_return_5d"]
]

X = df[feature_cols]
y = df["target_return_5d"]

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
alpha_values = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]

lasso_results = []

for alpha in alpha_values:

    lasso = Lasso(
        alpha=alpha,
        max_iter=10000
    )

    lasso.fit(X_train_scaled, y_train)

    coefficients = pd.Series(
        lasso.coef_,
        index=feature_cols
    )

    selected_features = coefficients[
        coefficients != 0
    ].index.tolist()

    lasso_results.append({
        "alpha": alpha,
        "n_selected_features": len(selected_features),
        "selected_features": selected_features
    })

lasso_feature_results = pd.DataFrame(lasso_results)

lasso_feature_results

,alpha,n_selected_features,selected_features
0,0.001,13,"[return_5d, intraday_return, high_low_range, s..."
1,0.010,0,[]
2,0.050,0,[]
3,0.100,0,[]
4,0.500,0,[]
5,1.000,0,[]


In [11]:
for alpha in [0.001, 0.01]:
    lasso = Lasso(
        alpha=alpha,
        max_iter=10000
    )
    
    lasso.fit(X_train_scaled, y_train)

    coefficients = pd.Series(
        lasso.coef_,
        index=feature_cols
    )

    print(f"\nalpha = {alpha}")
    print(coefficients.sort_values())


alpha = 0.001
atr_14             -0.016860
high_low_range     -0.007917
price_to_sma_5     -0.007444
return_5d          -0.001392
volatility_20      -0.000660
intraday_return    -0.000631
return_1d          -0.000000
volume_change_1d    0.000000
macd_signal         0.000000
roc_20              0.000000
roc_10             -0.000000
price_to_sma_60    -0.000000
sma_20              0.000000
sma_5               0.000000
gap                -0.000000
return_20d          0.000000
return_10d         -0.000000
price_to_sma_20     0.000000
volume_sma_20       0.001603
macd                0.002746
volatility_5        0.002843
macd_hist           0.002955
volume_ratio_20     0.007725
rsi_14              0.008230
sma_60              0.017975
dtype: float64

alpha = 0.01
return_1d           0.0
volume_change_1d    0.0
atr_14              0.0
volatility_20       0.0
volatility_5        0.0
macd_hist           0.0
macd_signal         0.0
macd                0.0
roc_20              0.0
roc_10         

In [12]:
print(y_train.describe())

count    2148.000000
mean        0.010328
std         0.075785
min        -0.311100
25%        -0.032081
50%         0.004312
75%         0.044833
max         0.461897
Name: target_return_5d, dtype: float64


In [13]:
X_train_scaled.shape
y_train.std()

np.float64(0.07578501306274806)

In [14]:
print(y_train.describe())
print(y_train.std())

count    2148.000000
mean        0.010328
std         0.075785
min        -0.311100
25%        -0.032081
50%         0.004312
75%         0.044833
max         0.461897
Name: target_return_5d, dtype: float64
0.07578501306274806


In [15]:
alpha_max = np.max(
    np.abs(X_train_scaled.T @ y_train)
) / len(X_train_scaled)

print("alpha_max:", alpha_max)

alpha_max: 0.0068210842145308885


In [16]:
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

alpha_values = [
    0.0001,
    0.0005,
    0.0010,
    0.0015,
    0.0020,
    0.0025,
    0.0030,
    0.0035,
    0.0040,
    0.0045,
    0.0050,
    0.0055,
    0.0060,
    0.0065,
    0.0068
]

lasso_results = []

for alpha in alpha_values:

    # 1. Lasso 학습
    lasso = Lasso(
        alpha=alpha,
        max_iter=10000
    )

    lasso.fit(X_train_scaled, y_train)

    # 2. coefficient 추출
    coefficients = pd.Series(
        lasso.coef_,
        index=feature_cols
    )

    # 3. coefficient != 0인 feature 선택
    selected_features = coefficients[
        coefficients != 0
    ].index.tolist()

    # 4. 선택된 feature가 없는 경우
    if len(selected_features) == 0:

        lasso_results.append({
            "alpha": alpha,
            "n_selected_features": 0,
            "selected_features": [],
            "MSE": np.nan,
            "RMSE": np.nan,
            "MAE": np.nan,
            "R2": np.nan
        })

        continue

    # 5. 선택된 feature의 column index
    selected_indices = [
        feature_cols.index(feature)
        for feature in selected_features
    ]

    X_train_selected = X_train_scaled[:, selected_indices]
    X_test_selected = X_test_scaled[:, selected_indices]

    # 6. 선택된 feature로 Lasso 예측
    lasso_selected = Lasso(
        alpha=alpha,
        max_iter=10000
    )

    lasso_selected.fit(
        X_train_selected,
        y_train
    )

    y_pred = lasso_selected.predict(
        X_test_selected
    )

    # 7. 성능 평가
    mse = mean_squared_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    r2 = r2_score(
        y_test,
        y_pred
    )

    # 8. 결과 저장
    lasso_results.append({
        "alpha": alpha,
        "n_selected_features": len(selected_features),
        "selected_features": selected_features,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

lasso_results_df = pd.DataFrame(lasso_results)

lasso_results_df

,alpha,n_selected_features,selected_features,MSE,RMSE,MAE,R2
0,0.0001,19,"[return_5d, return_10d, return_20d, intraday_r...",0.006339,0.079616,0.059382,0.049884
1,0.0005,14,"[return_5d, intraday_return, high_low_range, s...",0.006478,0.080487,0.059812,0.028982
2,0.0010,13,"[return_5d, intraday_return, high_low_range, s...",0.006604,0.081264,0.060239,0.010135
3,0.0015,10,"[return_5d, intraday_return, high_low_range, s...",0.006721,0.081981,0.060638,-0.007408
4,0.0020,8,"[intraday_return, high_low_range, sma_60, pric...",0.006707,0.081898,0.060511,-0.005373
5,0.0025,7,"[return_1d, high_low_range, price_to_sma_5, rs...",0.006697,0.081836,0.060433,-0.003837
6,0.0030,7,"[return_1d, high_low_range, price_to_sma_5, rs...",0.006686,0.081770,0.060344,-0.002233
7,0.0035,6,"[return_1d, price_to_sma_5, rsi_14, macd, macd...",0.006683,0.081750,0.060270,-0.001738
8,0.0040,4,"[rsi_14, macd, macd_hist, volume_ratio_20]",0.006682,0.081743,0.060227,-0.001548
9,0.0045,2,"[rsi_14, macd_hist]",0.006679,0.081727,0.060169,-0.001174


In [18]:
lasso_results_df[
    [
        "alpha",
        "n_selected_features",
        "selected_features",
        "MSE",
        "RMSE",
        "MAE",
        "R2"
    ]
]

,alpha,n_selected_features,selected_features,MSE,RMSE,MAE,R2
0,0.0001,19,"[return_5d, return_10d, return_20d, intraday_r...",0.006339,0.079616,0.059382,0.049884
1,0.0005,14,"[return_5d, intraday_return, high_low_range, s...",0.006478,0.080487,0.059812,0.028982
2,0.0010,13,"[return_5d, intraday_return, high_low_range, s...",0.006604,0.081264,0.060239,0.010135
3,0.0015,10,"[return_5d, intraday_return, high_low_range, s...",0.006721,0.081981,0.060638,-0.007408
4,0.0020,8,"[intraday_return, high_low_range, sma_60, pric...",0.006707,0.081898,0.060511,-0.005373
5,0.0025,7,"[return_1d, high_low_range, price_to_sma_5, rs...",0.006697,0.081836,0.060433,-0.003837
6,0.0030,7,"[return_1d, high_low_range, price_to_sma_5, rs...",0.006686,0.081770,0.060344,-0.002233
7,0.0035,6,"[return_1d, price_to_sma_5, rsi_14, macd, macd...",0.006683,0.081750,0.060270,-0.001738
8,0.0040,4,"[rsi_14, macd, macd_hist, volume_ratio_20]",0.006682,0.081743,0.060227,-0.001548
9,0.0045,2,"[rsi_14, macd_hist]",0.006679,0.081727,0.060169,-0.001174


In [20]:
output_path = "../../../data/processed/embedded_results/lasso_results.csv"

lasso_results_df.to_csv(
    output_path,
    index=False
)

print(f"Saved: {output_path}")

Saved: ../../../data/processed/embedded_results/lasso_results.csv


In [21]:
import os

print(os.path.exists(output_path))

True
